In [1]:
# Nivel 1 do desafio

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as map
import os
import json


In [3]:

os.listdir("/content")

['.config', 'dados_nivel_1.json', 'sample_data']

In [4]:
path_local = "../data/dados_nivel_1.json"
path_colab = "/content/dados_nivel_1.json"

In [5]:
#Lietura do arquivo josn
with open(path_colab, "r", encoding="utf-8") as f:
    dados =  json.load(f)

#a taxa de cambio fixa
taxa_cambio = dados["taxa_cambio_usd_brl"]

#convertando para dataframe
df = pd.DataFrame(dados['operacoes'])

print(taxa_cambio)
df.head()

5.4


,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           20 non-null     object
 1   cliente_id   20 non-null     object
 2   data         19 non-null     object
 3   valor        20 non-null     int64 
 4   moeda        20 non-null     object
 5   canal        20 non-null     object
 6   tipo         20 non-null     object
 7   contraparte  20 non-null     object
 8   observacao   20 non-null     object
dtypes: int64(1), object(8)
memory usage: 1.5+ KB


## Parte A - EDA

In [7]:
df

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [8]:
#passando data para o formato pandas
df["data"] = pd.to_datetime(df["data"])

In [9]:
#Encontrando e visualiando dados duplicados
df.duplicated()
df[df.duplicated()]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
9,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,


In [10]:
df = df.drop_duplicates()

In [11]:
df

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
10,OP-0010,CLI-A-4,2026-03-03,3800,BRL,cartao,pagamento,Alfa Comercio LTDA,


A operação id="OP-0013" fez uma transferencia de USD12k, ortanto precisaremos fazer a cnoversão entre moedas.  

In [12]:
valor_dol = df.loc[13]["valor"]
print(valor_dol)
valor_brl = round(taxa_cambio*valor_dol, 2)
print(valor_brl)
df.loc[13,"valor"] = valor_brl
df.loc[13, "moeda"] = "BRL"
df.loc[13]

12000
64800.0


,13
id,OP-0013
cliente_id,CLI-A-4
data,2026-03-24 00:00:00
valor,64800
moeda,BRL
canal,ted
tipo,transferencia_recebida
contraparte,Zeta Importacao
observacao,remessa internacional


Uma das linha não possui todas as informações necessárias -  sem informação da data da transação.

In [13]:
df[df["data"].isna()]

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
17,OP-0017,CLI-A-5,NaT,4300,BRL,especie,deposito,Gama Distribuidora,data nao capturada pelo sistema


A quantidade de transações por clientes

In [14]:
df['cliente_id'].value_counts()

,count
cliente_id,
CLI-A-1,4
CLI-A-4,4
CLI-A-5,4
CLI-A-3,3
CLI-A-2,2
CLI-A-6,2


In [15]:
df.sort_values(['cliente_id','valor'])

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,
10,OP-0010,CLI-A-4,2026-03-03,3800,BRL,cartao,pagamento,Alfa Comercio LTDA,


In [16]:
df.groupby('canal')['valor'].agg(
    ['count', 'sum', 'mean', 'min', 'max']
)

,count,sum,mean,min,max
canal,,,,,
boleto,3,11100,3700.0,2700,5100
cartao,2,5200,2600.0,1400,3800
especie,1,4300,4300.0,4300,4300
pix,8,101400,12675.0,2900,18100
ted,5,143500,28700.0,7000,64800


In [17]:
df.groupby('cliente_id')['valor'].agg(
    ['count', 'sum', 'mean', 'median', 'min', 'max']
)

,count,sum,mean,median,min,max
cliente_id,,,,,,
CLI-A-1,4,57500,14375.000000,17700.0,3300,18800
CLI-A-2,2,52900,26450.000000,26450.0,25900,27000
CLI-A-3,3,48500,16166.666667,16100.0,15200,17200
CLI-A-4,4,79500,19875.000000,5450.0,3800,64800
CLI-A-5,4,16900,4225.000000,3600.0,2700,7000
CLI-A-6,2,10200,5100.000000,5100.0,1400,8800


In [18]:
#Agrupando por valor total de transações por cliente
df.groupby("cliente_id")["valor"].sum()



,valor
cliente_id,
CLI-A-1,57500
CLI-A-2,52900
CLI-A-3,48500
CLI-A-4,79500
CLI-A-5,16900
CLI-A-6,10200


Analisando os canais que foram utilizadas pelos clientes

In [19]:
df.groupby("cliente_id")['canal'].value_counts()

cliente_id  canal  
CLI-A-1     pix        2
            boleto     1
            ted        1
CLI-A-2     ted        2
CLI-A-3     pix        3
CLI-A-4     boleto     1
            cartao     1
            pix        1
            ted        1
CLI-A-5     boleto     1
            especie    1
            pix        1
            ted        1
CLI-A-6     cartao     1
            pix        1
Name: count, dtype: int64

Os clientes costumam escolher o pix como forma de transação

Como data é um atributo importante para realizar a plicação das regras, então farei uma copia do df original e apagarei a linha que não possui essa informação.

In [20]:
df_cpy = df.dropna(subset=["data"])


In [21]:
fracionamento = df_cpy.groupby(['cliente_id', 'data']).agg(
    qtd_datas=('data','count'),
    soma_valores=('valor', 'sum')
)

fracionamento['fracionamento'] = (fracionamento['qtd_datas'] >=3) & (fracionamento['soma_valores'] >= 50000)
print(fracionamento)

                       qtd_datas  soma_valores  fracionamento
cliente_id data                                              
CLI-A-1    2026-03-09          3         54200           True
           2026-03-21          1          3300          False
CLI-A-2    2026-03-14          2         52900          False
CLI-A-3    2026-03-05          3         48500          False
CLI-A-4    2026-03-03          1          3800          False
           2026-03-11          1          5100          False
           2026-03-18          1          5800          False
           2026-03-24          1         64800          False
CLI-A-5    2026-03-07          1          2900          False
           2026-03-16          1          7000          False
           2026-03-26          1          2700          False
CLI-A-6    2026-03-12          1          8800          False
           2026-03-28          1          1400          False


Os clientes cujos id são ('CLI-A-1', 'CLI-A-3) foram os únicos a realizarem 3 transações num mesmo dia, porém somente o primeiro ('CLI-A-1') ultrapassou o limite de 50k.

In [22]:
df_cpy =  df_cpy.merge(fracionamento[['fracionamento']], on=['cliente_id', 'data'], how='left')
df_cpy

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,fracionamento
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,True
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,True
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,,True
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,,False
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,,False
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,,False
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,,False
9,OP-0010,CLI-A-4,2026-03-03,3800,BRL,cartao,pagamento,Alfa Comercio LTDA,,False


In [23]:
df_cpy['qtd_operacoes'] = (
    df_cpy.groupby('cliente_id')['cliente_id']
    .transform('count')
)

df_cpy['media_cliente'] = (
    df_cpy.groupby('cliente_id')['valor']
    .transform('median')
)

df_cpy['mediana_cliente'] = (
    df_cpy.groupby('cliente_id')['valor']
    .transform('mean')
)

df_cpy['sup'] = (
    (df_cpy['qtd_operacoes'] >= 4) &
    (df_cpy['valor'] >= df_cpy['mediana_cliente'] * 5)
)

df_cpy

,id,cliente_id,data,valor,moeda,canal,tipo,contraparte,observacao,fracionamento,qtd_operacoes,media_cliente,mediana_cliente,sup
0,OP-0001,CLI-A-1,2026-03-09,18100,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,True,4,17700.0,14375.000000,False
1,OP-0002,CLI-A-1,2026-03-09,17300,BRL,pix,transferencia_enviada,Alfa Comercio LTDA,,True,4,17700.0,14375.000000,False
2,OP-0003,CLI-A-1,2026-03-09,18800,BRL,ted,transferencia_enviada,Beta Servicos ME,,True,4,17700.0,14375.000000,False
3,OP-0004,CLI-A-1,2026-03-21,3300,BRL,boleto,pagamento,Gama Distribuidora,,False,4,17700.0,14375.000000,False
4,OP-0005,CLI-A-2,2026-03-14,25900,BRL,ted,transferencia_enviada,Delta Transportes,,False,2,26450.0,26450.000000,False
5,OP-0006,CLI-A-2,2026-03-14,27000,BRL,ted,transferencia_enviada,Delta Transportes,,False,2,26450.0,26450.000000,False
6,OP-0007,CLI-A-3,2026-03-05,17200,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False,3,16100.0,16166.666667,False
7,OP-0008,CLI-A-3,2026-03-05,15200,BRL,pix,transferencia_enviada,Epsilon Consultoria,,False,3,16100.0,16166.666667,False
8,OP-0009,CLI-A-3,2026-03-05,16100,BRL,pix,transferencia_enviada,Zeta Importacao,,False,3,16100.0,16166.666667,False
9,OP-0010,CLI-A-4,2026-03-03,3800,BRL,cartao,pagamento,Alfa Comercio LTDA,,False,4,5450.0,19875.000000,False


Ao analisarmos o dataframe, nota-se que a regra funcionou, pois no id = OP-0013:

qtd_operações = 4

Valor = 64800

Mediana = 5450 -> 5*Mediana = 27250

logo, 64,8K é um valor atipico segundo as regras

## Parte B - Análise com LLM

In [24]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [25]:
dados = df_cpy.loc[12].to_dict()

prompt = f"""
Faça uma análise dos dados deste cliente e classifique:

- nivel_risco: baixo, médio ou alto
- tipologia_suspeita: sim ou não
- red_flags: lista de sinais de alerta
- justificativa: explique todas as decisões tomadas

Dados da operação:

{json.dumps(dados, ensure_ascii=False, indent=2, default=str)}

A resposta deve vir em formato JSON.
"""
messages = [
    {"role": "user", "content": prompt}
]


text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    do_sample=False
)

input_length = inputs["input_ids"].shape[1]

resposta = tokenizer.decode(
    outputs[0][input_length:],
    skip_special_tokens=True
)

print(resposta)

```json
{
  "nivel_risco": "baixo",
  "tipologia_suspeita": "não",
  "red_flags": [],
  "justificativa": {
    "nivel_risco": "O valor da operação é considerado baixo em comparação com o histórico do cliente. O valor total da operação é R$ 64,800, que é menor que a média e mediana históricas do cliente.",
    "tipologia_suspeita": "Não há indícios de tipologia suspeita. A contraparte é uma empresa conhecida (Zeta Importação) e a operação é uma transferência recebida.",
    "red_flags": "Nenhum sinal de alerta foi identificado durante a análise.",
    "conclusao": "A operação está dentro dos padrões aceitáveis e não apresenta riscos significativos."
  }
}
```


In [26]:

dados = df_cpy.loc[9].to_dict()

prompt2 = f"""
Diante das informações apresentadas, determine as seguintes propriedades:

1. nivel_risco: baixo, médio ou alto
2. tipologia_suspeita: sim ou não
3. red_flags: possíveis sinais de alerta
4. justificativa: explique objetivamente sua decisão.

Dados da operação:

{json.dumps(dados, ensure_ascii=False, indent=2, default=str)}

A resposta deve vir em formato JSON.
"""

messages = [
    {"role": "user", "content": prompt2}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=300,
    do_sample=False
)

input_length = inputs["input_ids"].shape[1]

resposta = tokenizer.decode(
    outputs[0][input_length:],
    skip_special_tokens=True
)

print(resposta)

```json
{
  "nivel_risco": "baixo",
  "tipologia_suspeita": "não",
  "red_flags": [
    {
      "sinal_de_alerta": "Quantidade de operações alta (4)",
      "justificativa": "O cliente realizou quatro transações no mesmo dia, o que pode indicar uma atividade mais frequente e potencialmente normal para alguns clientes."
    },
    {
      "sinal_de_alerta": "Valor médio do cliente alto (5450.0 BRL)",
      "justificativa": "O valor médio do cliente é considerado alto, mas isso não necessariamente indica risco, pois pode ser um padrão comum para algumas empresas."
    }
  ],
  "justificativa": "A operação tem um valor médio elevado, mas a quantidade de transações é considerada normal para alguns clientes. Não há outros sinais de alerta que levem a uma classificação de alto risco. Portanto, a classificação é de nível baixo."
}
```
